# UAV Small Object Detection - Google Colab Training

This notebook trains the enhanced YOLOv8 model on VisDrone dataset using free Google Colab GPU.

**Configured for your setup:**
- Repo: `https://github.com/Naimur-Rahman-Asif/uav_small_object_detection.git`
- Dataset: `/content/drive/MyDrive/uav_small_object_detection/data/VisDrone/`

**Setup Steps:**
1. Runtime → Change runtime type → GPU (T4)
2. Run all cells in order
3. Model checkpoints/results are saved to Google Drive

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/uav_small_object_detection/outputs', exist_ok=True)
print('✓ Google Drive mounted')

In [ ]:
# Check GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ No GPU detected. Go to Runtime → Change runtime type → GPU')

In [ ]:
# Clone project from GitHub (configured for your repository)
!git clone https://github.com/Naimur-Rahman-Asif/uav_small_object_detection.git
%cd /content/uav_small_object_detection

# Link dataset directly from Google Drive (avoids copying large files)
!rm -rf data/VisDrone
!ln -s /content/drive/MyDrive/uav_small_object_detection/data/VisDrone data/VisDrone

print('✓ Project cloned and dataset linked from Drive')

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q ultralytics pyyaml tqdm numpy opencv-python pillow matplotlib scipy wandb

print('✓ Dependencies installed')

In [ ]:
# Verify VisDrone dataset is present
import os
from pathlib import Path

data_path = Path('data/VisDrone')
splits = ['train', 'val', 'test']

for split in splits:
    img_dir = data_path / split / 'images'
    ann_dir = data_path / split / 'annotations'
    
    if img_dir.exists() and ann_dir.exists():
        img_count = len(list(img_dir.glob('*.jpg')))
        ann_count = len(list(ann_dir.glob('*.txt')))
        print(f'✓ {split}: {img_count} images, {ann_count} annotations')
    else:
        print(f'⚠️ {split}: NOT FOUND')
        print(f'   Please upload VisDrone dataset to data/VisDrone/{split}/')

In [ ]:
# Use cloud-optimized config
!cat configs/train_config_cloud.yaml

In [ ]:
# Optional: Setup Weights & Biases for experiment tracking
# !pip install -q wandb
# import wandb
# wandb.login()  # Enter your API key when prompted

# Apply Colab-safe training config (prevents common CUDA OOM on T4)
import yaml
from pathlib import Path

config_path = Path('configs/train_config_cloud.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

safe_overrides = {
    'use_wandb': False,
    'model_scale': 'n',
    'batch_size': 2,
    'gradient_accumulation_steps': 4,
    'img_size': 512,
    'mixed_precision': True,
    'num_workers': 2,
    'pin_memory': True,
    'drop_last': True,
    'lr': 5e-4
}
config.update(safe_overrides)

with open(config_path, 'w') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print('✓ Applied safe cloud config:')
for key in ['model_scale', 'batch_size', 'gradient_accumulation_steps', 'img_size', 'mixed_precision', 'num_workers', 'pin_memory', 'drop_last', 'lr', 'use_wandb']:
    print(f'  {key}: {config.get(key)}')

In [ ]:
# Start training (with CUDA allocator setting to reduce fragmentation OOM)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python main.py train --config configs/train_config_cloud.yaml --device cuda

In [ ]:
# Copy trained weights/results to Google Drive for persistence
!mkdir -p /content/drive/MyDrive/uav_small_object_detection/outputs
!cp -r weights /content/drive/MyDrive/uav_small_object_detection/outputs/
!cp -r results /content/drive/MyDrive/uav_small_object_detection/outputs/
print('✓ Weights and results saved to Google Drive outputs folder')

## Testing & Inference

In [ ]:
# Test on validation set
!python main.py validate --model weights/best_model.pth --config configs/train_config_cloud.yaml --device cuda

In [ ]:
# Run inference on a single image
!python main.py infer --model weights/best_model.pth --image data/VisDrone/test/images/0000006_00159_d_0000001.jpg --device cuda

## Download Results

Results are automatically saved to Google Drive. You can also download them directly:

In [ ]:
# Zip and download weights
!zip -r weights.zip weights/
from google.colab import files
files.download('weights.zip')